## FuRBO

### 1. Introdução

O Feasibility-Driven Trust Region Bayesian Optimization (FuRBO) foi proposto em 2025 por Ascia et al. (Universidade Técnica de Munique / Universidade de Leiden), como uma extensão do algoritmo SCBO (Scalable Constrained Bayesian Optimization). O FuRBO foi especialmente concebido para resolver problemas de otimização com restrições caras (black-box), em que a região factível pode ser extremamente pequena e difícil de encontrar, especialmente em espaços de alta dimensionalidade.

A ideia central é redefinir a Trust Region (TR) de forma que ela seja guiada não apenas pelo melhor ponto avaliado até o momento, mas também pela informação dos modelos surrogate de objetivo e restrições — usando um mecanismo de **inspetores** que amostram o espaço ao redor do melhor candidato e votam, por ranking de viabilidade e qualidade, em qual sub-região deve ser explorada a seguir.

### 2. O Problema

FuRBO resolve o problema de minimização com restrições black-box:

$$
x^* = \arg\min_{x \in \Omega} f(x) \quad \text{sujeito a} \quad c_k(x) \leq 0, \; \forall k \in \{1, \ldots, K\}
$$

em que $\Omega \subset \mathbb{R}^D$ é o espaço de busca, $f$ é a função objetivo e $c_k$ são as funções de restrição — todas avaliadas apenas por consultas pontuais (sem expressão analítica conhecida). O conjunto factível é definido como:

$$
\Omega_{\text{feas}} = \{ x \in \Omega \mid c_k(x) \leq 0, \; \forall k \}
$$

O desafio é que, em problemas de alta dimensão com restrições severas, $\Omega_{\text{feas}}$ pode ser uma região minúscula dentro de $\Omega$, tornando extremamente difícil encontrar sequer uma única solução factível dentro do orçamento de avaliações.

### 3. O Algoritmo

#### 3.1 Base: SCBO

FuRBO herda a estrutura do SCBO (Eriksson & Poloczek, 2021), que por sua vez é uma extensão do TuRBO para o caso com restrições. No SCBO:

- Modelos GP independentes são treinados para o objetivo $f$ e para cada restrição $c_k$.
- Uma Trust Region (TR) é inicializada centrada no melhor ponto factível avaliado (ou no ponto de menor violação, caso nenhum factível tenha sido encontrado ainda).
- O próximo batch de candidatos é selecionado via **Thompson Sampling** dentro da TR, considerando ao mesmo tempo as predições do GP do objetivo e das restrições.
- A TR cresce se houver muitos sucessos consecutivos (melhora do melhor valor) e encolhe se houver muitos fracassos.

A limitação do SCBO é que ele usa apenas o melhor ponto avaliado para **posicionar** a TR, sem explorar ativamente o que os surrogates preveem sobre a localização da região factível.

#### 3.2 A inovação do FuRBO: Inspetores e TR guiada por viabilidade

A principal diferença do FuRBO está na forma como a Trust Region é **construída** a cada iteração. Em vez de simplesmente centrar a TR no melhor ponto e expandir/contrair uniformemente, o FuRBO faz o seguinte:

**Passo 1 — Ranking das amostras avaliadas.**  
Todos os pontos avaliados são ordenados: os factíveis pelo valor do objetivo $f(x)$ (menor é melhor), e os infactíveis pela violação máxima normalizada:
$$
v(x) = \max_{k} \frac{c_k(x)}{\max|c_k(x)|}
$$
Os factíveis sempre precedem os infactíveis nessa lista.

**Passo 2 — Geração dos Inspetores.**  
Em torno do melhor candidato $x_{best}$, são amostrados uniformemente $N$ pontos dentro de uma bola de raio $R$:
$$
\mathcal{I} = \{ x_1, \ldots, x_N \} \sim \text{Uniform}\left( \mathcal{B}(x_{best}, R) \right)
$$
Esses pontos **não são avaliados** na função real — eles são apenas amostrados e depois ranqueados pelos modelos surrogate.

**Passo 3 — Ranking dos Inspetores pelos Surrogates.**  
Os inspetores são ranqueados da mesma forma que as amostras avaliadas, mas usando as **predições** dos GPs do objetivo e das restrições, sem chamar as funções reais. Isso é computacionalmente barato.

**Passo 4 — Definição da Trust Region.**  
Os $P\%$ melhores inspetores (chamados de $\mathcal{I}_{best}$) são selecionados. A TR é então definida como o **menor hiperretângulo** que contém todos os pontos em $\mathcal{I}_{best}$:
$$
\text{TR} = \prod_{j=1}^{D} \left[ \min_{x \in \mathcal{I}_{best}} x_j, \; \max_{x \in \mathcal{I}_{best}} x_j \right]
$$
Esse mecanismo faz com que a TR se mova e se redimensione livremente pelo espaço, guiada pelo que os surrogates preveem como as regiões mais promissoras e factíveis — sem depender exclusivamente da posição do melhor ponto avaliado.

**Passo 5 — Thompson Sampling dentro da TR.**  
O próximo batch de $q$ candidatos é proposto via Thompson Sampling sobre os GPs de objetivo e restrições, restrito à TR recém-definida.

**Passo 6 — Atualização do raio $R$.**  
Contadores de sucesso ($n_s$) e fracasso ($n_f$) monitoram o progresso. Se $n_s = \tau_s$, o raio $R$ é dobrado; se $n_f = \tau_f$, o raio é reduzido à metade. Se $R \leq \varepsilon$, o algoritmo reinicia.

#### 3.3 Por que isso funciona?

A vantagem do mecanismo de inspetores é que ele permite que a TR **salte** para regiões distantes do melhor ponto avaliado, caso os surrogates prevejam que a região factível está em outro lugar. Isso é especialmente poderoso quando:

- A região factível é muito pequena e o ponto atual ainda é infactível — a TR consegue se deslocar em direção à fronteira de factibilidade predita.
- O espaço tem muitas dimensões — a TR pode ter formato não-quadrado, se adaptando ao contorno previsto da região factível.
- Há múltiplos ótimos locais — os inspetores ajudam o algoritmo a escapar de regiões ruins.

No SCBO, a TR sempre está centrada no melhor ponto atual — se esse ponto estiver longe da região factível real, a TR inteira pode estar numa zona inútil. No FuRBO, a TR se move para onde os surrogates dizem que vale a pena olhar.

### 4. Implementação

A seguir, implementamos o FuRBO do zero usando PyTorch e BoTorch, e o testamos em um problema de otimização com restrições: minimização da **função Sphere em 10D** com uma restrição linear que reduz severamente o espaço factível.

In [ ]:
import math
import warnings
from dataclasses import dataclass

import gpytorch
import torch
from gpytorch.constraints import Interval
from gpytorch.kernels import MaternKernel, ScaleKernel
from gpytorch.likelihoods import GaussianLikelihood
from gpytorch.mlls import ExactMarginalLogLikelihood
from torch.quasirandom import SobolEngine

from botorch.fit import fit_gpytorch_mll
from botorch.generation import MaxPosteriorSampling
from botorch.models import SingleTaskGP

warnings.filterwarnings("ignore")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
dtype = torch.double

#### 4.1 Problema de teste: Sphere 10D com restrição

Usaremos a função Sphere em 10 dimensões com uma restrição linear severa:

$$
f(x) = \sum_{i=1}^{D} x_i^2, \quad x \in [-5, 10]^D
$$

Restrição (deixa apenas ~10% do espaço factível):

$$
c(x) = \sum_{i=1}^{D} x_i - 4D \leq 0
$$

O ótimo global factível está próximo de $x^* = (0, 0, \ldots, 0)$, que satisfaz $c(x^*) = 0 \leq 0$.

In [ ]:
DIM = 10
LB = -5.0
UB = 10.0

def objective(x: torch.Tensor) -> torch.Tensor:
    """Sphere: minimizar sum(x_i^2). Retorna negado pois BoTorch maximiza."""
    x_unnorm = LB + (UB - LB) * x  # desfaz normalização [0,1]^D -> [LB, UB]^D
    return -torch.sum(x_unnorm ** 2, dim=-1)

def constraint(x: torch.Tensor) -> torch.Tensor:
    """c(x) = sum(x_i) - 4*D <= 0. Retorna o valor da restrição (<=0 é factível)."""
    x_unnorm = LB + (UB - LB) * x
    return torch.sum(x_unnorm, dim=-1) - 4.0 * DIM

#### 4.2 Estado do FuRBO

In [ ]:
@dataclass
class FuRBOState:
    """Estado do FuRBO: rastreia o raio R dos inspetores e contadores de sucesso/fracasso."""
    dim: int
    batch_size: int
    radius: float = 1.0          # R inicial: cobre todo o domínio normalizado
    radius_min: float = 5e-8     # limiar de reinício
    success_counter: int = 0
    failure_counter: int = 0
    success_tolerance: int = 2   # tau_s
    failure_tolerance: int = 3   # tau_f
    best_value: float = -float("inf")  # melhor f(x) factível (negado)
    best_constraint: float = float("inf")  # menor violação de restrição
    restart_triggered: bool = False


def update_furbo_state(state: FuRBOState,
                       Y_next: torch.Tensor,
                       C_next: torch.Tensor) -> FuRBOState:
    """Atualiza contadores e raio R baseado no novo batch avaliado."""
    feasible_mask = (C_next <= 0).squeeze()
    improved = False

    if feasible_mask.any():
        best_new = Y_next[feasible_mask].max().item()
        if best_new > state.best_value + 1e-3 * abs(state.best_value):
            state.best_value = best_new
            improved = True
    else:
        min_violation = C_next.min().item()
        if min_violation < state.best_constraint - 1e-6:
            state.best_constraint = min_violation
            improved = True

    if improved:
        state.success_counter += 1
        state.failure_counter = 0
    else:
        state.success_counter = 0
        state.failure_counter += 1

    if state.success_counter >= state.success_tolerance:
        state.radius = min(state.radius * 2.0, 1.0)
        state.success_counter = 0
    elif state.failure_counter >= state.failure_tolerance:
        state.radius /= 2.0
        state.failure_counter = 0

    if state.radius < state.radius_min:
        state.restart_triggered = True

    return state

#### 4.3 Ranking por viabilidade e objetivo

In [ ]:
def rank_points(Y: torch.Tensor, C: torch.Tensor) -> torch.Tensor:
    """
    Retorna índices ordenados pelo critério do FuRBO:
    - Factíveis (C <= 0) primeiro, ranqueados por Y decrescente (maior Y = menor f real).
    - Infactíveis depois, ranqueados por violação normalizada crescente.
    """
    feasible = (C.squeeze() <= 0)
    infeasible = ~feasible
    ranked_idx = []

    if feasible.any():
        feas_idx = feasible.nonzero(as_tuple=True)[0]
        order = Y[feas_idx].squeeze().argsort(descending=True)
        ranked_idx.append(feas_idx[order])

    if infeasible.any():
        infeas_idx = infeasible.nonzero(as_tuple=True)[0]
        violations = C.squeeze()[infeas_idx]
        max_v = violations.abs().max().clamp(min=1e-8)
        norm_v = violations / max_v
        order = norm_v.argsort()
        ranked_idx.append(infeas_idx[order])

    return torch.cat(ranked_idx)

#### 4.4 Geração de inspetores e definição da Trust Region

In [ ]:
def sample_inspectors(x_best: torch.Tensor, radius: float,
                      n_inspectors: int = 1000) -> torch.Tensor:
    """
    Amostra n_inspectors pontos uniformemente dentro da bola B(x_best, R)
    no espaço normalizado [0,1]^D.
    """
    dim = x_best.shape[0]
    z = torch.randn(n_inspectors, dim, dtype=dtype, device=device)
    z = z / z.norm(dim=-1, keepdim=True).clamp(min=1e-8)
    # Escala uniforme dentro da bola: u^(1/D) onde u ~ Uniform[0,1]
    r = radius * (torch.rand(n_inspectors, 1, dtype=dtype, device=device) ** (1.0 / dim))
    inspectors = x_best.unsqueeze(0) + r * z
    return inspectors.clamp(0.0, 1.0)


def define_trust_region(inspectors_best: torch.Tensor):
    """
    Define a TR como o menor hiperretângulo contendo os melhores inspetores.
    Retorna (tr_lb, tr_ub) cada um de shape [D].
    """
    tr_lb = inspectors_best.min(dim=0).values
    tr_ub = inspectors_best.max(dim=0).values
    eps = 1e-4
    mask = (tr_ub - tr_lb) < eps
    tr_lb[mask] = (tr_lb[mask] - eps).clamp(0.0, 1.0)
    tr_ub[mask] = (tr_ub[mask] + eps).clamp(0.0, 1.0)
    return tr_lb, tr_ub

#### 4.5 Geração do batch via Thompson Sampling e ajuste dos GPs

In [ ]:
def fit_gp(X: torch.Tensor, Y: torch.Tensor) -> SingleTaskGP:
    """Ajusta um GP com kernel Matern 5/2 aos dados (X, Y)."""
    Y_norm = (Y - Y.mean()) / (Y.std() + 1e-8)
    likelihood = GaussianLikelihood(noise_constraint=Interval(1e-8, 1e-3))
    covar_module = ScaleKernel(
        MaternKernel(nu=2.5, ard_num_dims=X.shape[-1],
                     lengthscale_constraint=Interval(0.005, 4.0))
    )
    model = SingleTaskGP(X, Y_norm, covar_module=covar_module, likelihood=likelihood)
    mll = ExactMarginalLogLikelihood(model.likelihood, model)
    with gpytorch.settings.max_cholesky_size(float("inf")):
        fit_gpytorch_mll(mll)
    return model


def generate_furbo_batch(
    state: FuRBOState,
    model_obj: SingleTaskGP,
    model_con: SingleTaskGP,
    X: torch.Tensor,
    Y: torch.Tensor,
    C: torch.Tensor,
    batch_size: int,
    inspector_pct: float = 0.10,
    n_inspectors: int = 1000,
    n_candidates: int = 5000,
) -> torch.Tensor:
    """
    Gera o próximo batch de candidatos usando o mecanismo de inspetores do FuRBO.
    """
    # Passo 1: Melhor candidato atual pelo ranking de viabilidade
    ranked_idx = rank_points(Y, C)
    x_best = X[ranked_idx[0]]

    # Passo 2: Gera inspetores ao redor de x_best
    inspectors = sample_inspectors(x_best, state.radius, n_inspectors)

    # Passo 3: Rankeia inspetores pelos surrogates (sem avaliar a função real)
    with torch.no_grad():
        pred_Y = model_obj.posterior(inspectors).mean
        pred_C = model_con.posterior(inspectors).mean

    ranked_insp = rank_points(pred_Y, pred_C)
    n_best = max(2, int(inspector_pct * n_inspectors))
    best_insp = inspectors[ranked_insp[:n_best]]

    # Passo 4: Define a TR como o menor hiperretângulo em torno dos melhores inspetores
    tr_lb, tr_ub = define_trust_region(best_insp)

    # Passo 5: Thompson Sampling dentro da TR
    sobol = SobolEngine(DIM, scramble=True)
    cands = sobol.draw(n_candidates).to(dtype=dtype, device=device)
    cands = tr_lb + (tr_ub - tr_lb) * cands

    thompson_sampling = MaxPosteriorSampling(model=model_obj, replacement=False)
    with torch.no_grad():
        X_next = thompson_sampling(cands, num_samples=batch_size)

    return X_next

#### 4.6 Loop principal do FuRBO

In [ ]:
dim = DIM
batch_size = dim          # q = D (como no artigo)
n_init = 3 * dim          # DoE inicial: 3D pontos
n_total = 30 * dim        # Orçamento total: 30D avaliações

torch.manual_seed(42)

# Inicialização com sequência de Sobol
sobol = SobolEngine(dimension=dim, scramble=True, seed=42)
X_furbo = sobol.draw(n_init).to(dtype=dtype, device=device)
Y_furbo = objective(X_furbo).unsqueeze(-1)
C_furbo = constraint(X_furbo).unsqueeze(-1)

state = FuRBOState(dim=dim, batch_size=batch_size)
feasible_init = (C_furbo <= 0).squeeze()
if feasible_init.any():
    state.best_value = Y_furbo[feasible_init].max().item()

print(f"Inicialização: {n_init} pontos, factíveis: {feasible_init.sum().item()}/{n_init}")
print(f"Estado inicial: raio R={state.radius:.2e}\n")

# Loop de otimização
while len(X_furbo) < n_total and not state.restart_triggered:
    model_obj = fit_gp(X_furbo, Y_furbo)
    model_con = fit_gp(X_furbo, C_furbo)

    X_next = generate_furbo_batch(
        state=state, model_obj=model_obj, model_con=model_con,
        X=X_furbo, Y=Y_furbo, C=C_furbo, batch_size=batch_size,
    )

    Y_next = objective(X_next).unsqueeze(-1)
    C_next = constraint(X_next).unsqueeze(-1)

    state = update_furbo_state(state, Y_next, C_next)

    X_furbo = torch.cat([X_furbo, X_next], dim=0)
    Y_furbo = torch.cat([Y_furbo, Y_next], dim=0)
    C_furbo = torch.cat([C_furbo, C_next], dim=0)

    feasible_all = (C_furbo <= 0).squeeze()
    if feasible_all.any():
        best_f = -Y_furbo[feasible_all].max().item()
        print(f"{len(X_furbo):4d}) Melhor f(x) factível: {best_f:.4e}  "
              f"| Factíveis: {feasible_all.sum().item():3d}/{len(X_furbo)}  "
              f"| Raio R: {state.radius:.2e}")
    else:
        print(f"{len(X_furbo):4d}) Sem factível ainda  "
              f"| Menor violação: {C_furbo.min().item():.4e}  "
              f"| Raio R: {state.radius:.2e}")

#### 4.7 Resultado final

In [ ]:
feasible_all = (C_furbo <= 0).squeeze()

if feasible_all.any():
    best_idx = Y_furbo[feasible_all].argmax()
    x_best_final = X_furbo[feasible_all][best_idx]
    f_best = -Y_furbo[feasible_all][best_idx].item()
    c_best = C_furbo[feasible_all][best_idx].item()
    x_unnorm = LB + (UB - LB) * x_best_final

    print(f"=== Melhor solução factível encontrada ===")
    print(f"f(x*)   = {f_best:.6f}  (ótimo global factível ≈ 0.0)")
    print(f"c(x*)   = {c_best:.6f}  (<= 0 é factível)")
    print(f"||x*||  = {x_unnorm.norm().item():.4f}")
    print(f"\nTotal de avaliações usadas: {len(X_furbo)} / {n_total}")
    print(f"Pontos factíveis encontrados: {feasible_all.sum().item()}")
else:
    print("Nenhuma solução factível encontrada dentro do orçamento.")

### 5. Comparação FuRBO vs SCBO (simplificado)

Para ilustrar a diferença conceitual, implementamos uma versão simplificada do SCBO (sem o mecanismo de inspetores — a TR é simplesmente um quadrado centrado no melhor ponto) e comparamos com o FuRBO no mesmo problema.

In [ ]:
def generate_scbo_batch(
    X: torch.Tensor, Y: torch.Tensor, C: torch.Tensor,
    model_obj: SingleTaskGP, batch_size: int,
    tr_length: float = 0.8, n_candidates: int = 5000,
) -> torch.Tensor:
    """SCBO simplificado: TR centrada no melhor ponto, sem inspetores."""
    ranked_idx = rank_points(Y, C)
    x_best = X[ranked_idx[0]]
    half = tr_length / 2.0
    tr_lb = (x_best - half).clamp(0.0, 1.0)
    tr_ub = (x_best + half).clamp(0.0, 1.0)

    sobol = SobolEngine(DIM, scramble=True)
    cands = sobol.draw(n_candidates).to(dtype=dtype, device=device)
    cands = tr_lb + (tr_ub - tr_lb) * cands

    thompson_sampling = MaxPosteriorSampling(model=model_obj, replacement=False)
    with torch.no_grad():
        return thompson_sampling(cands, num_samples=batch_size)


# Execução comparativa
torch.manual_seed(42)
sobol = SobolEngine(dimension=dim, scramble=True, seed=42)
X_scbo = sobol.draw(n_init).to(dtype=dtype, device=device)
Y_scbo = objective(X_scbo).unsqueeze(-1)
C_scbo = constraint(X_scbo).unsqueeze(-1)

torch.manual_seed(42)
sobol2 = SobolEngine(dimension=dim, scramble=True, seed=42)
X_f2 = sobol2.draw(n_init).to(dtype=dtype, device=device)
Y_f2 = objective(X_f2).unsqueeze(-1)
C_f2 = constraint(X_f2).unsqueeze(-1)
state2 = FuRBOState(dim=dim, batch_size=batch_size)

best_scbo_hist, best_furbo_hist = [], []
n_iterations = (n_total - n_init) // batch_size

for it in range(n_iterations):
    # SCBO
    m_obj_s = fit_gp(X_scbo, Y_scbo)
    X_n_s = generate_scbo_batch(X_scbo, Y_scbo, C_scbo, m_obj_s, batch_size)
    X_scbo = torch.cat([X_scbo, X_n_s])
    Y_scbo = torch.cat([Y_scbo, objective(X_n_s).unsqueeze(-1)])
    C_scbo = torch.cat([C_scbo, constraint(X_n_s).unsqueeze(-1)])
    feas_s = (C_scbo <= 0).squeeze()
    best_scbo_hist.append(-Y_scbo[feas_s].max().item() if feas_s.any() else float('nan'))

    # FuRBO
    if not state2.restart_triggered:
        m_obj_f = fit_gp(X_f2, Y_f2)
        m_con_f = fit_gp(X_f2, C_f2)
        X_n_f = generate_furbo_batch(state2, m_obj_f, m_con_f,
                                      X_f2, Y_f2, C_f2, batch_size)
        Y_n_f = objective(X_n_f).unsqueeze(-1)
        C_n_f = constraint(X_n_f).unsqueeze(-1)
        state2 = update_furbo_state(state2, Y_n_f, C_n_f)
        X_f2 = torch.cat([X_f2, X_n_f])
        Y_f2 = torch.cat([Y_f2, Y_n_f])
        C_f2 = torch.cat([C_f2, C_n_f])
    feas_f = (C_f2 <= 0).squeeze()
    best_furbo_hist.append(-Y_f2[feas_f].max().item() if feas_f.any() else float('nan'))

print("Comparação finalizada!")

In [ ]:
import matplotlib.pyplot as plt

evals = [n_init + (i + 1) * batch_size for i in range(n_iterations)]

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(evals, best_furbo_hist, color='darkorange', linewidth=2.0, label='FuRBO')
ax.plot(evals, best_scbo_hist, color='forestgreen', linewidth=2.0, label='SCBO (simplificado)')
ax.axhline(0.0, color='gray', linestyle='--', linewidth=1.0, label='Ótimo global (f*=0)')
ax.set_xlabel('Avaliações', fontsize=12)
ax.set_ylabel('Melhor f(x) factível', fontsize=12)
ax.set_title(f'FuRBO vs SCBO — Sphere {dim}D com restrição severa', fontsize=13)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### 6. Limitações e Trabalhos Futuros

Conforme apontado no artigo original, o FuRBO tem algumas limitações importantes:

1. **Custo computacional adicional**: o mecanismo de inspetores introduz chamadas extras aos modelos surrogate a cada iteração, tornando o algoritmo mais caro que o SCBO. Isso é justificável apenas quando as avaliações da função real são dominantes em tempo.

2. **Restrições muito severas em alta dimensão**: em problemas com dezenas de restrições ativas em 40+ dimensões, ambos FuRBO e SCBO frequentemente falham em encontrar regiões factíveis dentro do orçamento disponível.

3. **Trust Region única e alinhada aos eixos**: a TR atual é um único hiperretângulo alinhado aos eixos. Versões futuras poderiam explorar múltiplas TRs ou TRs com orientações adaptativas.

4. **Sensibilidade ao percentual $P\%$**: o percentual de inspetores selecionados para definir a TR influencia significativamente o desempenho e pode precisar de ajuste por problema. O artigo aponta que percentuais menores (e.g. 1%) convergem mais rápido inicialmente, mas podem limitar a exploração de regiões factíveis diversas.